In [1]:
"""
Single-face verification using RetinaFace.

Usage:
    python single_face_verifier.py --image path/to/image.png

This script:
  - runs RetinaFace on the image
  - verifies that exactly ONE face is detected
  - optionally checks that the face is within a given ROI and has reasonable size
  - prints a simple JSON-like report
"""

import argparse
import json
from typing import List, Tuple, Dict, Any
import os
import random
import cv2
from retinaface import RetinaFace

In [2]:
def detect_faces(image_path: str,
                 score_thresh: float = 0.9) -> List[Dict[str, Any]]:
    """
    Run RetinaFace on an image and return a list of detections.

    Each detection dict contains:
        - 'score' : float
        - 'box'   : (x1, y1, x2, y2)  pixel coordinates
    """
    # RetinaFace.detect_faces returns a dict keyed by face index or 'face_1', etc.
    detections_raw = RetinaFace.detect_faces(image_path)

    faces = []
    if isinstance(detections_raw, dict):
        for key, det in detections_raw.items():
            score = float(det.get("score", 0.0))
            if score < score_thresh:
                continue
            x1, y1, x2, y2 = det["facial_area"]
            faces.append(
                {
                    "score": score,
                    "box": (int(x1), int(y1), int(x2), int(y2)),
                }
            )
    return faces


def bbox_area(box: Tuple[int, int, int, int]) -> int:
    """Compute area of a bounding box (x1, y1, x2, y2)."""
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_center(box: Tuple[int, int, int, int]) -> Tuple[float, float]:
    """Compute center of a bounding box."""
    x1, y1, x2, y2 = box
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0


def verify_single_face(
    image_path: str,
    score_thresh: float = 0.9,
    min_rel_area: float = 0.02,
    max_rel_area: float = 0.6,
    roi: Tuple[float, float, float, float] | None = None,
) -> Dict[str, Any]:
    """
    Verify that there is exactly one face in the image, with reasonable size and position.

    Args:
        image_path: path to the image.
        score_thresh: minimum RetinaFace confidence to keep a detection.
        min_rel_area: minimum face area as fraction of full image area.
        max_rel_area: maximum face area as fraction of full image area.
        roi: optional region of interest as (x_min_rel, y_min_rel, x_max_rel, y_max_rel)
             in normalized coordinates [0,1]. The face center must lie inside if given.

    Returns:
        dict with:
            - 'ok': bool
            - 'reason': str
            - 'num_faces': int
            - 'faces': list of detections (score, box)
    """
    # Load image to get dimensions
    img = cv2.imread(image_path)
    if img is None:
        return {
            "ok": False,
            "reason": "Image could not be loaded",
            "num_faces": 0,
            "faces": [],
        }

    h, w = img.shape[:2]
    img_area = float(h * w)

    # Run RetinaFace detection
    faces = detect_faces(image_path, score_thresh=score_thresh)
    num_faces = len(faces)

    # Basic count checks
    if num_faces == 0:
        return {
            "ok": False,
            "reason": "No faces detected above threshold",
            "num_faces": 0,
            "faces": faces,
        }

    if num_faces > 1:
        return {
            "ok": False,
            "reason": f"Multiple faces detected ({num_faces})",
            "num_faces": num_faces,
            "faces": faces,
        }

    # Now we know we have exactly one face
    face = faces[0]
    box = face["box"]

    # Area check
    fa = bbox_area(box)
    rel_area = fa / img_area if img_area > 0 else 0.0

    if rel_area < min_rel_area:
        return {
            "ok": False,
            "reason": f"Face too small (relative area={rel_area:.4f})",
            "num_faces": num_faces,
            "faces": faces,
        }

    if rel_area > max_rel_area:
        return {
            "ok": False,
            "reason": f"Face too large (relative area={rel_area:.4f})",
            "num_faces": num_faces,
            "faces": faces,
        }

    # ROI check (optional)
    if roi is not None:
        x_min_rel, y_min_rel, x_max_rel, y_max_rel = roi
        cx, cy = bbox_center(box)
        cx_rel = cx / w
        cy_rel = cy / h

        if not (x_min_rel <= cx_rel <= x_max_rel and
                y_min_rel <= cy_rel <= y_max_rel):
            return {
                "ok": False,
                "reason": (
                    f"Face center ({cx_rel:.3f},{cy_rel:.3f}) "
                    f"outside ROI {roi}"
                ),
                "num_faces": num_faces,
                "faces": faces,
            }

    # If we reach here, all checks passed
    return {
        "ok": True,
        "reason": "Exactly one face with acceptable size and position",
        "num_faces": num_faces,
        "faces": faces,
    }




In [3]:
def _collect_images(root):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}
    imgs = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in exts:
                imgs.append(os.path.join(dirpath, fn))
    return imgs

def sample_pos_neg(n, base_dir='.', pos_sub='data/Original', neg_sub='data/random_doc_images', seed=None):
    """
    Return two lists: (positive_samples, negative_samples).
    - n: number of samples per class (if fewer available, returns all).
    - base_dir: base path containing the data folders (default: current dir).
    - seed: optional int for deterministic sampling.
    """
    if seed is not None:
        random.seed(seed)

    pos_root = os.path.join(base_dir, pos_sub)
    neg_root = os.path.join(base_dir, neg_sub)

    pos_imgs = _collect_images(pos_root)
    neg_imgs = _collect_images(neg_root)

    if not pos_imgs:
        raise FileNotFoundError(f"No positive images found in {pos_root}")
    if not neg_imgs:
        raise FileNotFoundError(f"No negative images found in {neg_root}")

    k_pos = min(n, len(pos_imgs))
    k_neg = min(n, len(neg_imgs))

    pos_sample = random.sample(pos_imgs, k_pos)
    neg_sample = random.sample(neg_imgs, k_neg)

    return pos_sample, neg_sample


In [6]:
# Example ROI for a driving licence photo:
# left 0–0.35 of width, middle 0.15–0.85 of height
roi = (0.0, 0.15, 0.35, 0.85)

pos, neg = sample_pos_neg(10, base_dir='.', seed=42)

sample_paths = pos + neg

for path in sample_paths:
    print("=" * 80)
    print(f"Verifying image: {path}")
    result = verify_single_face(
        image_path=path,
        score_thresh=0.9,
        min_rel_area=0.02,
        max_rel_area=0.6,
        roi=roi,
    )
    # Pretty-print as JSON
    print("result:", result)

Verifying image: .\data/Original\generated_license_656.png
26-01-11 19:06:02 - Directory C:\Users\rahma/.deepface created
26-01-11 19:06:02 - Directory C:\Users\rahma/.deepface/weights created
26-01-11 19:06:02 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: C:\Users\rahma\.deepface\weights\retinaface.h5
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119M/119M [00:02<00:00, 40.8MB/s]


result: {'ok': False, 'reason': 'Face too small (relative area=0.0013)', 'num_faces': 1, 'faces': [{'score': 0.993256688117981, 'box': (147, 265, 178, 304)}]}
Verifying image: .\data/Original\generated_license_1408.png
result: {'ok': True, 'reason': 'Exactly one face with acceptable size and position', 'num_faces': 1, 'faces': [{'score': 0.9945567846298218, 'box': (92, 314, 227, 543)}]}
Verifying image: .\data/Original\generated_license_109.png
result: {'ok': True, 'reason': 'Exactly one face with acceptable size and position', 'num_faces': 1, 'faces': [{'score': 0.9978671073913574, 'box': (107, 297, 260, 557)}]}
Verifying image: .\data/Original\generated_license_2010.png
result: {'ok': False, 'reason': 'No faces detected above threshold', 'num_faces': 0, 'faces': []}
Verifying image: .\data/Original\generated_license_1900.png
result: {'ok': True, 'reason': 'Exactly one face with acceptable size and position', 'num_faces': 1, 'faces': [{'score': 0.9994693398475647, 'box': (0, 202, 387,